# Prédiction de la taxonomie pour des séquences 
Ce notebook vise à présenter l'utilisation des modèles XGBoost entraînés afin d'effectuer une classification hiérarchique des séquences de génomes. Pour ce faire, vous avez besoin de :
 1. un répertoire contenant un arbre phylogénétique `phylo_tree.txt`  et des modèles xgboost entrainés pour chaque noeud interne de l'arbre, au format `.json` comme par exmple `Bacillales_order.json`. Ces fichiers sont générés lors de l'entrainement sur un dataset
 2. de paramètres d'inférence regroupés dans le yaml `predict_params.yaml`
 3. d'un fichier .fna ou d'un répertoire contenant des génomes à classifier. Un fichier .fna peut contenir plusieurs séquences qui seront chacune classifiées


In [7]:
import sys
sys.path.append('../..')
from wisp_light.prediction.predict import TaxoPredictor

In [8]:
model_dir = "/home/hcourtei/Projects/MicroTaxo/codes/exp/model_base_complete_05_15_16_37"
predict_dir = None  # permet de sauvegarder les log et les predictions
params_file = "/home/hcourtei/Projects/MicroTaxo/codes/wisp_light/prediction/predict_params.yaml"
fna_path = "/home/hcourtei/Projects/MicroTaxo/codes/data/refseq_data/GCF_000725405.1_ASM72540v1_genomic.fna"

Dans la paramètrisation donnée par le yaml, on échantillone dans la séquence `max_sampling=100` fenêtres de taille `read_size= 10_000` , sur chacune desquelles on dénombre  les kmers de taille `ksize=4`

 
## 1. Explication d'une Prédiction brute

In [9]:
predictor = TaxoPredictor(model_dir, params_file, predict_dir=predict_dir)
all_results = predictor.predict_one_fna(fna_path, raw_pred=True,verbose=True)


None
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - TaxoPredictor init for XGBoost models from directory: /home/hcourtei/Projects/MicroTaxo/codes/exp/model_base_complete_05_15_16_37
16:15 - TaxoPredictor - INFO - TaxoPredictor init for XGBoost models from directory: /home/hcour

Dans la prédiction brute, 'Root': {'Actinomycetota': 84} signifie que 84 fenêtres sont classifiés 'Actinomycetota'
et les autres ??

## 2. Résumé dans un tableau

In [10]:
print(predictor.results_table.to_markdown())

|    | fna_file                               | id_seq        | phylum                   | class                    | order                                         | family                                         |
|---:|:---------------------------------------|:--------------|:-------------------------|:-------------------------|:----------------------------------------------|:-----------------------------------------------|
|  0 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008889.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 95, 'Actinomycetales': 1}] | [{'Dermacoccaceae': 88, 'Micrococcaceae': 1}]  |
|  1 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008890.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 35, 'Actinomycetales': 6}] | [{'Dermacoccaceae': 45, 'Micrococcaceae': 26}] |


## 3. Extraction de la Prediction majoritaire

In [11]:
all_results = predictor.predict_one_fna(fna_path, raw_pred=False, verbose=False)
print(predictor.results_table.to_markdown())

|    | fna_file                               | id_seq        | phylum                   | class                    | order                                         | family                                         |
|---:|:---------------------------------------|:--------------|:-------------------------|:-------------------------|:----------------------------------------------|:-----------------------------------------------|
|  0 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008889.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 95, 'Actinomycetales': 1}] | [{'Dermacoccaceae': 88, 'Micrococcaceae': 1}]  |
|  1 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008890.1 | [{'Actinomycetota': 84}] | [{'Actinomycetes': 100}] | [{'Micrococcales': 35, 'Actinomycetales': 6}] | [{'Dermacoccaceae': 45, 'Micrococcaceae': 26}] |
|  2 | GCF_000725405.1_ASM72540v1_genomic.fna | NZ_CP008889.1 | Actinomycetota           | Actinomycetes            | Micrococcales         

## 4. Prediction sur un répertoire de fichiers .fna
Une mention "Seq too short"  est faite sur la classification lorsque les séquences sont trop courtes , ici < `read_size= 10_000`

In [12]:
fna_dir = "/home/hcourtei/Projects/MicroTaxo/codes/predict/to_predict"
predictor = TaxoPredictor(model_dir, params_file, predict_dir=None)
predictor.predict_one_dir(fna_dir, raw_pred=False, verbose=False)
print(predictor.results_table.to_markdown())

None
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - TaxoPredictor - INFO - Params for prediction
{'ksize': 4,
 'max_sampling': 100,
 'normalisation_func': 'delta_mean',
 'pattern': [1, 1, 1, 1],
 'read_identity_threshold': 0.8,
 'read_size': 10000,
 'shift_ratio': None,
 'threshold': 0.6}
16:15 - Tax

100%|██████████| 5/5 [00:18<00:00,  3.76s/it]

|    | fna_file                               | id_seq            | phylum         | class               | order             | family             |
|---:|:---------------------------------------|:------------------|:---------------|:--------------------|:------------------|:-------------------|
|  0 | GCF_000737865.1_ASM73786v1_genomic.fna | NZ_CP007287.1     | Actinomycetota | Actinomycetes       | Bifidobacteriales | Bifidobacteriaceae |
|  1 | GCF_000741575.1_Bifcun_genomic.fna     | NZ_JGYV01000001.1 | Actinomycetota | Actinomycetes       | Bifidobacteriales | Bifidobacteriaceae |
|  2 | GCF_000741575.1_Bifcun_genomic.fna     | NZ_JGYV01000002.1 | Actinomycetota | Actinomycetes       | Bifidobacteriales | Bifidobacteriaceae |
|  3 | GCF_000741575.1_Bifcun_genomic.fna     | NZ_JGYV01000003.1 | Read_too_small | Read_too_small      | Read_too_small    | Read_too_small     |
|  4 | GCF_000741575.1_Bifcun_genomic.fna     | NZ_JGYV01000004.1 | Actinomycetota | Actinomycetes       | Bifid